In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = (
    SparkSession.builder
    .appName("MinIO-PostgreSQL")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars", "/home/jovyan/postgresql-42.7.1.jar")  # PostgreSQL JDBC driver
    .getOrCreate()
)

hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [37]:
bronze="s3a://lazimsiz/bronze"
bronze_delta="s3a://lazimsiz/bronze_delta"


In [32]:
import os
fp='/home/jovyan/work'
os.listdir(fp)
##containerdeki fayllari gormek ucun

['spark_code.ipynb', 'files', 'test_scripts']

In [12]:
df_card=spark.read.csv('/home/jovyan/work/files/card.csv',header=True,inferSchema=True)
df_customer=spark.read.csv('/home/jovyan/work/files/customer.csv',header=True,inferSchema=True)
df_trs=spark.read.csv('/home/jovyan/work/files/trs.csv',header=True,inferSchema=True)

In [29]:
import pandas as pd
from IPython.display import FileLink
df = pd.read_csv('/home/jovyan/work/files/trs.csv')
filename = 'trs_json.json'
df.to_json(filename, orient='records')
display(FileLink(filename))

/home/jovyan/trs_json.json

In [33]:
df_json_spark_test=spark.read.format('json').load('/home/jovyan/work/test_scripts/trs_json.json')

In [ ]:
df_card.write.mode("overwrite").option("header","true").csv(f"{bronze}/card.csv")
df_trs.write.mode("overwrite").option("header","true").csv(f"{bronze}/trs.csv")
df_customer.write.mode("overwrite").option("header","true").csv(f"{bronze}/customer.csv")
##burda adi fayl kimi bronze buckete yukleyirem

In [39]:
df_card.write.mode("overwrite").format("delta").save(f"{bronze_delta}/card")

In [ ]:
df_card.write.mode("overwrite").format("delta").save(f"{bronze_delta}/card")
df_trs.write.mode("overwrite").format("delta").save(f"{bronze_delta}/trs")
df_customer.write.mode("overwrite").format("delta").save(f"{bronze_delta}/customer")
##burda delta kimi yazdim


In [ ]:
#